# Stage 1 — Baseline Regression Models

Naive baselines, Linear Regression, Random Forest, XGBoost, and a BPNN (MLP) for next-day rainfall regression on the locked chronological split.

## 1. Imports

In [3]:
import pandas as pd
import numpy as np
import json, time, warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import os

## 2. Load data & build the next-day rainfall target

Predicts **tomorrow's** rainfall from **today's** features, dropping rows at series-end or date-gap boundaries.

In [4]:
t0 = time.time()
df = pd.read_csv("../Data/Processed/Processed.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["Station", "date"]).reset_index(drop=True)

# --- Correct the target: predict TOMORROW's rainfall from TODAY's features ---
df["Target_Rainfall"] = df.groupby("Station")["BMD_Rainfall"].shift(-1)
df["next_date"] = df.groupby("Station")["date"].shift(-1)
gap_days = (df["next_date"] - df["date"]).dt.days
valid = df["Target_Rainfall"].notna() & (gap_days == 1)
print(f"Dropped rows (series end / gap boundary): {len(df) - valid.sum()}")
df = df[valid].drop(columns=["next_date"]).reset_index(drop=True)

Dropped rows (series end / gap boundary): 37


## 3. Locked chronological split

Train ≤ 2007, validate 2008–2011, test ≥ 2012 — kept identical across every stage script so results are comparable.

In [5]:
feature_cols = [
    "Year", "Month", "DOY", "T2M", "RH2M", "PS", "WS2M", "CLOUD_AMT",
    "ALLSKY_SFC_SW_DWN", "T2MDEW", "Rain_lag1", "Rain_lag2", "Rain_7day_avg",
    "Temp_lag1", "Humidity_lag1", "Pressure_lag1", "Station_Mean_Rainfall",
]

train = df[df["Year"] <= 2007]
val   = df[(df["Year"] >= 2008) & (df["Year"] <= 2011)]
test  = df[df["Year"] >= 2012]
print(f"train={len(train)}  val={len(val)}  test={len(test)}")

X_train, y_train = train[feature_cols], train["Target_Rainfall"]
X_val, y_val = val[feature_cols], val["Target_Rainfall"]
X_test, y_test = test[feature_cols], test["Target_Rainfall"]

results = {}

def score(name, pred):
    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae = float(mean_absolute_error(y_test, pred))
    r2 = float(r2_score(y_test, pred))
    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}
    print(f"{name:32s} RMSE={rmse:7.3f}  MAE={mae:6.3f}  R2={r2:7.4f}   ({time.time()-t0:6.1f}s)")

train=286109  val=49672  test=62084


## 4. Naive baselines

Persistence (tomorrow = today) and seasonal climatology (per-station, per-month mean, fit on train only).

In [6]:
# Persistence: tomorrow = today's rainfall
score("Naive - Persistence", test["BMD_Rainfall"].values)

# Seasonal climatology: per-station, per-month mean rainfall, computed from TRAIN years only
clim = train.groupby(["Station", "Month"])["Target_Rainfall"].mean().rename("clim").reset_index()
test_clim = test.merge(clim, on=["Station", "Month"], how="left")
test_clim["clim"] = test_clim["clim"].fillna(train["Target_Rainfall"].mean())
score("Naive - Seasonal climatology", test_clim["clim"].values)

Naive - Persistence              RMSE= 20.362  MAE= 7.633  R2=-0.1684   (   2.2s)
Naive - Seasonal climatology     RMSE= 17.441  MAE= 8.088  R2= 0.1428   (   2.3s)


## 5. Linear Regression

In [7]:
scaler = StandardScaler().fit(X_train)
lr = LinearRegression().fit(scaler.transform(X_train), y_train)
score("Linear Regression", lr.predict(scaler.transform(X_test)))

Linear Regression                RMSE= 15.736  MAE= 8.047  R2= 0.3022   (   2.9s)


## 6. Random Forest

In [8]:
rf = RandomForestRegressor(n_estimators=300, max_depth=14, min_samples_leaf=5,
                            n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
score("Random Forest", rf.predict(X_test))

Random Forest                    RMSE= 14.438  MAE= 6.289  R2= 0.4126   ( 330.4s)


## 7. XGBoost

In [9]:
xgb_model = xgb.XGBRegressor(
    n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, random_state=42,
    early_stopping_rounds=30, eval_metric="rmse",
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
score("XGBoost", xgb_model.predict(X_test))

XGBoost                          RMSE= 14.315  MAE= 6.160  R2= 0.4225   ( 338.6s)


## 8. Back-propagation Neural Network (paper 1's BPNN, via `MLPRegressor`)

In [10]:
mlp = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                    alpha=1e-4, batch_size=512, learning_rate_init=1e-3,
                    max_iter=60, early_stopping=True, n_iter_no_change=8,
                    random_state=42)
mlp.fit(scaler.transform(X_train), y_train)
score("Back-propagation NN (MLP)", mlp.predict(scaler.transform(X_test)))

Back-propagation NN (MLP)        RMSE= 14.401  MAE= 6.042  R2= 0.4156   ( 374.7s)


## 9. Save results

In [16]:
os.makedirs("../outputs", exist_ok=True)
pd.DataFrame(results).T.to_csv("../outputs/model-results/stage1_baseline_regression_models_results.csv")
print("done stage1", time.time() - t0)

done stage1 592.2186605930328
